#### Understanding `model.py`

This section explains how the `NeuralNet` in `src/model.py` is structured and how it implements
a fully-connected neural network (MLP) for MNIST.

**Model architecture**

We use a multilayer perceptron with this layer size sequence:

\[
784 \rightarrow 128 \rightarrow 64 \rightarrow 10
\]

- 784 = 28×28 input pixels (flattened).
- 128, 64 = hidden layer widths.
- 10 = number of classes (digits 0–9).

The model applies:
- Linear → ReLU → Linear → ReLU → Linear
- Optionally softmax on the final logits to get probabilities.

##### 1. Importing the model and recap of the API

The key components in `model.py` are:

- `relu(x)`: elementwise ReLU activation.
- `softmax(x)`: numerically stable softmax along the class dimension.
- `NeuralNet`:
  - constructor: defines layer dimensions and initializes weights and biases,
  - `forward(X, apply_softmax=False)`: runs the forward pass through all layers.

We now import `NeuralNet` and inspect its basic behavior.

In [ ]:
from pathlib import Path
import sys

import numpy as np

# Ensure src/ is on sys.path
project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from model import NeuralNet

##### 2. Constructor: defining the function family

The `NeuralNet` constructor sets up the *family* of functions we are working with,
by deciding:

- input dimension `input_dim` (default 784),
- hidden layer dimensions `hidden_dims` (default (128, 64)),
- output dimension `output_dim` (default 10).

It then builds:

- `layer_dims = [input_dim, *hidden_dims, output_dim]`,
- `weights`: a list of weight matrices,
- `biases`: a list of bias vectors.

For the default MNIST case:
- `layer_dims = [784, 128, 64, 10]`
- `weights[0]` has shape `(784, 128)`
- `weights[1]` has shape `(128, 64)`
- `weights[2]` has shape `(64, 10)`

Each `(in_dim, out_dim)` pair corresponds to one linear layer: \(z = a W + b\).

In [ ]:
model = NeuralNet()

print("layer_dims:", model.layer_dims)
print("num_layers:", model.num_layers)

for i, (W, b) in enumerate(zip(model.weights, model.biases)):
    print(f"Layer {i}: W.shape={W.shape}, b.shape={b.shape}")

##### 3. Forward pass: chaining linear transforms and ReLU

The `forward` method implements the computation:

\[
A_0 = X \\
Z_1 = A_0 W_1 + b_1,\quad A_1 = \mathrm{ReLU}(Z_1) \\
Z_2 = A_1 W_2 + b_2,\quad A_2 = \mathrm{ReLU}(Z_2) \\
Z_3 = A_2 W_3 + b_3 \quad (\text{logits})
\]

If `apply_softmax=True`, we further compute:

\[
\hat{Y} = \mathrm{softmax}(Z_3)
\]

Key points:

- Hidden layers apply `linear → ReLU`.
- The last layer applies only `linear` and produces logits of shape `(batch_size, 10)`.
- ReLU introduces nonlinearity so the model can learn non-linear decision boundaries.
- Softmax converts logits into probabilities over the 10 classes.

In [ ]:
BATCH_SIZE = 4
X_fake = np.random.rand(BATCH_SIZE, 784).astype(np.float32)

logits = model.forward(X_fake)
probs = model.forward(X_fake, apply_softmax=True)

print("Input shape:", X_fake.shape)
print("Logits shape:", logits.shape)
print("Probs shape:", probs.shape)
print("Probs row sums:", probs.sum(axis=1))

##### 4. Internal caching: preparing for backprop

During the forward pass, the model stores intermediate values in `self._cache`:

- `activations`: `[A0, A1, A2]`
- `pre_activations`: `[Z1, Z2]`
- `logits`: `[Z3]`

These are exactly the values needed later for backpropagation, when we apply the chain rule:

- The derivative of ReLU at Z depends on where Z > 0.
- The gradient flowing into each layer depends on activations from the previous layer.

By caching them, `backward` can recompute gradients without rerunning the whole forward pass from scratch.

In [ ]:
# Run a forward pass to populate cache
_ = model.forward(X_fake)

cache = model._cache
print("Cached keys:", cache.keys())

activations = cache["activations"]
pre_activations = cache["pre_activations"]
logits_cached = cache["logits"][0]

print("Number of activations:", len(activations))
for i, A in enumerate(activations):
    print(f"A{i}.shape =", A.shape)

print("Number of pre-activations:", len(pre_activations))
for i, Z in enumerate(pre_activations, start=1):
    print(f"Z{i}.shape =", Z.shape)

print("logits_cached.shape:", logits_cached.shape)